# Spouse Recommendation System - Hargeisa, Somaliland
### Based on the MSc Thesis by Abdiwasac Arab Abdillahi (2026)

This notebook implements the **Spouse Recommendation System** described in the thesis. It uses a **Gradient Boosting** model trained on socio-cultural and economic factors to predict marital stability and inverts that prediction to generate a **Compatibility Score (0-100%)**.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

## 1. Synthetic Data Generation
Since the original dataset is private, we generate synthetic data (n=150) that follows the distributions and correlations reported in the thesis (e.g., Table 7, Table 8, and Table 11).

In [ ]:
def generate_synthetic_data(n=150):
    # Target: 57.3% Married (0), 42.7% Divorced (1)
    target = np.random.choice([0, 1], size=n, p=[0.573, 0.427])
    
    data = pd.DataFrame({'Marital_Status': target})
    
    # Cultural Factors (1-5 Likert Scale) - Strong correlation with target
    # Spouse Choice Autonomy (Negative correlation: -0.52)
    data['Spouse_Choice_Autonomy'] = data['Marital_Status'].apply(lambda x: np.random.randint(3, 6) if x == 0 else np.random.randint(1, 4))
    
    # Clan Approval Requirement (Positive correlation: +0.51)
    data['Clan_Approval_Requirement'] = data['Marital_Status'].apply(lambda x: np.random.randint(1, 4) if x == 0 else np.random.randint(3, 6))
    
    # Parental Influence (Positive correlation: +0.49)
    data['Parental_Influence'] = data['Marital_Status'].apply(lambda x: np.random.randint(1, 4) if x == 0 else np.random.randint(3, 6))
    
    # Demographic Factors
    # Age at Marriage (Negative correlation: -0.43)
    data['Age_at_Marriage'] = data['Marital_Status'].apply(lambda x: np.random.normal(24.8, 4.5) if x == 0 else np.random.normal(21.1, 4.2))
    
    # Current Age
    data['Age'] = data['Age_at_Marriage'] + np.random.randint(1, 20)
    
    # Education Level (0-5)
    data['Education_Level'] = data['Marital_Status'].apply(lambda x: np.random.choice([3, 4, 5], p=[0.3, 0.5, 0.2]) if x == 0 else np.random.choice([0, 1, 2, 3], p=[0.2, 0.3, 0.3, 0.2]))
    
    # Economic Factors
    # Income Category (0=Low, 1=Middle, 2=High)
    data['Income_Category'] = data['Marital_Status'].apply(lambda x: np.random.choice([1, 2], p=[0.6, 0.4]) if x == 0 else np.random.choice([0, 1], p=[0.6, 0.4]))
    
    # Employment Status (0=Employed, 1=Self, 2=Unemployed)
    data['Employment_Status'] = data['Marital_Status'].apply(lambda x: np.random.choice([0, 1], p=[0.8, 0.2]) if x == 0 else np.random.choice([1, 2], p=[0.4, 0.6]))
    
    # Clan Type (0=Different, 1=Same)
    data['Clan_Type'] = np.random.choice([0, 1], size=n, p=[0.55, 0.45])
    
    # Gender (0=Male, 1=Female)
    data['Gender'] = np.random.choice([0, 1], size=n)
    
    return data

df = generate_synthetic_data()
print("Dataset Preview:")
display(df.head())

## 2. Model Training (Gradient Boosting)
Following the thesis methodology: SMOTE for balancing and Gradient Boosting for classification.

In [ ]:
# Features and Target
X = df.drop('Marital_Status', axis=1)
y = df['Marital_Status']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Balance with SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

# Train Gradient Boosting Model (Optimal parameters from Table 5)
model = GradientBoostingClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5, 
    subsample=0.8, 
    random_state=42
)
model.fit(X_train_scaled, y_train_res)

print(f"Model Training Complete. Test Accuracy: {model.score(X_test_scaled, y_test):.2f}")

## 3. Recommendation Logic (Inversion of Prediction)
We implement the **Feature Integration Rules** from Table 6 to create a synthetic couple profile and then calculate compatibility.

In [ ]:
def calculate_compatibility(person_a, person_b):
    """
    Calculates compatibility score between two individuals based on thesis Table 6.
    """
    # 1. Create Synthetic Couple Profile
    couple_profile = {
        'Gender': person_a['Gender'], # Arbitrary for the model report, usually filtered by M/F
        'Age': (person_a['Age'] + person_b['Age']) / 2,
        'Age_at_Marriage': min(person_a['Age'], person_b['Age']), # Youngest partner immaturity
        'Education_Level': min(person_a['Education_Level'], person_b['Education_Level']),
        'Employment_Status': max(person_a['Employment_Status'], person_b['Employment_Status']), # Max vulnerability
        'Income_Category': max(person_a['Income_Category'], person_b['Income_Category']), # Pooled resources
        'Clan_Type': 1 if person_a['Clan'] == person_b['Clan'] else 0,
        'Spouse_Choice_Autonomy': min(person_a['Autonomy'], person_b['Autonomy']), # Forced union risk
        'Parental_Influence': max(person_a['Parental_Influence'], person_b['Parental_Influence']), # External interference
        'Clan_Approval_Requirement': max(person_a['Clan_Approval'], person_b['Clan_Approval']) # Structural pressure
    }
    
    # Convert to DataFrame for prediction
    profile_df = pd.DataFrame([couple_profile])[X.columns]
    profile_scaled = scaler.transform(profile_df)
    
    # 2. Predict Divorce Probability
    # index 1 is probability of divorce (class 1)
    p_divorce = model.predict_proba(profile_scaled)[0][1]
    
    # 3. Invert to Compatibility Score
    compatibility = 100 * (1 - p_divorce)
    
    # 4. Categorize (Section 3.11.4)
    category = ""
    if compatibility >= 85: category = "Very High Compatibility"
    elif compatibility >= 70: category = "Good Compatibility"
    elif compatibility >= 50: category = "Moderate Compatibility"
    else: category = "Low Compatibility"
    
    return round(compatibility, 2), category

# Example Profiles
ali = {'Gender': 0, 'Age': 28, 'Education_Level': 4, 'Employment_Status': 0, 'Income_Category': 2, 'Clan': 'A', 'Autonomy': 5, 'Parental_Influence': 1, 'Clan_Approval': 1}
hawa = {'Gender': 1, 'Age': 24, 'Education_Level': 4, 'Employment_Status': 0, 'Income_Category': 1, 'Clan': 'A', 'Autonomy': 5, 'Parental_Influence': 2, 'Clan_Approval': 1}

score, cat = calculate_compatibility(ali, hawa)
print(f"Recommendation Result for Ali & Hawa:")
print(f"Score: {score}% - {cat}")

## 4. Visualizing Feature Importance
As shown in Figure 8 of the thesis, cultural factors dominate the system.

In [ ]:
importances = model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Feature Importance in Spouse Recommendation (Gradient Boosting)')
plt.show()